In [1]:
import os
import json
import numpy as np
import h5py
from scipy.spatial import cKDTree

In [ ]:
PFIELD_FOLDER = "/home/dysco/Neeraj/neuralop_project/data/raw data/poisson_dataset1/input/"
FDOM_FOLDER   = "/home/dysco/Neeraj/neuralop_project/data/raw data/poisson_dataset1/output/"
PFIELD_BASENAME = "airfoil_pfield"
FDOM_BASENAME   = "airfoil_fdom_fdom"
OUT_FOLDER = "training_pairs_clean"
FRAME_START = 100
FRAME_STOP  = 400
FRAME_STEP  = 10

In [3]:
os.makedirs(OUT_FOLDER, exist_ok=True)

def read_required_pfield(pfile):
    """Read deterministic pfield datasets. Raise if missing or shapes mismatch."""
    with h5py.File(pfile, "r") as hf:
        needed = ["X", "Gamma", "sigma", "circulation", "vol", "static", "i", "C"]
        for k in needed:
            if k not in hf:
                raise KeyError(f"Dataset '{k}' not found in {pfile}")
        pos = np.array(hf["X"])
        Gamma_vec = np.array(hf["Gamma"])
        sigma = np.array(hf["sigma"])
        circulation = np.array(hf["circulation"])
        vol = np.array(hf["vol"])
        static = np.array(hf["static"])
        idx_i = np.array(hf["i"])
        C = np.array(hf["C"])

    # Basic shape checks
    N = pos.shape[0]
    if pos.ndim != 2 or pos.shape[1] != 3:
        raise ValueError(f"pos shape must be (N,3); got {pos.shape} in {pfile}")
    if Gamma_vec.shape[0] != N:
        raise ValueError(f"Gamma first dim must match N (pos). Got {Gamma_vec.shape} vs N={N}")
    if Gamma_vec.ndim == 1 and Gamma_vec.size == 3*N:
        Gamma_vec = Gamma_vec.reshape(N,3)
    if Gamma_vec.ndim != 2 or Gamma_vec.shape[1] not in (2,3):
        raise ValueError(f"Gamma must be (N,3) or (N,2). Got {Gamma_vec.shape} in {pfile}")

    # Ensure scalar arrays length N
    for name, arr in [("sigma", sigma), ("circulation", circulation), ("vol", vol), ("static", static), ("i", idx_i)]:
        arr = np.array(arr)
        if arr.shape[0] != N:
            raise ValueError(f"Field '{name}' length must be N={N}. Got {arr.shape} in {pfile}")
        # replace variable with standardized array
        if name == "sigma":
            sigma = arr
        elif name == "circulation":
            circulation = arr
        elif name == "vol":
            vol = arr
        elif name == "static":
            static = arr
        elif name == "i":
            idx_i = arr

    if C.shape[0] != N or C.ndim !=2 or C.shape[1] not in (2,3):
        raise ValueError(f"C must be (N,3) or (N,2). Got {C.shape} in {pfile}")

    # gamma_scalar: derive robustly (use magnitude of Gamma_vec)
    if Gamma_vec.shape[1] == 2:
        # 2D vector -> scalar vorticity may be the perpendicular component; use norm as default
        gamma_scalar = np.linalg.norm(Gamma_vec, axis=1)
    else:
        gamma_scalar = np.linalg.norm(Gamma_vec, axis=1)

    return {
        "pos": pos.astype(np.float64),
        "Gamma_vec": Gamma_vec.astype(np.float64),
        "gamma_scalar": gamma_scalar.astype(np.float64),
        "sigma": sigma.astype(np.float64),
        "circulation": circulation.astype(np.float64),
        "vol": vol.astype(np.float64),
        "static": static.astype(np.int32),
        "i": idx_i.astype(np.int32),
        "C": C.astype(np.float64),
    }

In [4]:
def read_required_fdom(ffile):
    """Read deterministic fdom datasets (nodes, U). Flatten U if grid-shaped."""
    with h5py.File(ffile, "r") as hf:
        # Prefer 'nodes' and 'U' exact names
        if "nodes" not in hf and "X" not in hf:
            # try 'nodes' alternatives; but in this clean loader we enforce exact naming
            raise KeyError(f"'nodes' not found in {ffile}")
        nodes_key = "nodes" if "nodes" in hf else "X"
        if "U" not in hf:
            raise KeyError(f"'U' not found in {ffile}")
        nodes = np.array(hf[nodes_key])
        U_arr = np.array(hf["U"])

    # normalize nodes shape to (M,3)
    if nodes.ndim == 2 and nodes.shape[1] in (2,3):
        nodes = nodes.astype(np.float64)
    else:
        # attempt to flatten last dim if e.g., nodes shaped (nx,ny,nz,3)
        if nodes.ndim >= 3 and nodes.shape[-1] in (2,3):
            nodes = nodes.reshape(-1, nodes.shape[-1]).astype(np.float64)
        else:
            raise ValueError(f"nodes shape not recognized: {nodes.shape} in {ffile}")

    # normalize U to (M,2|3)
    if U_arr.ndim == 4 and U_arr.shape[-1] in (2,3):
        U_nodes = U_arr.reshape(-1, U_arr.shape[-1]).astype(np.float64)
    elif U_arr.ndim == 2 and U_arr.shape[1] in (2,3):
        U_nodes = U_arr.astype(np.float64)
    else:
        # if U is flattened 1D or other, attempt reshape if counts match
        if U_arr.size == nodes.shape[0] * (3 if nodes.shape[1]==3 else nodes.shape[1]):
            U_nodes = U_arr.reshape(nodes.shape[0], -1).astype(np.float64)
        else:
            raise ValueError(f"U shape not recognized or does not match nodes: U{U_arr.shape}, nodes{nodes.shape} in {ffile}")

    if nodes.shape[0] != U_nodes.shape[0]:
        raise ValueError(f"Number of nodes ({nodes.shape[0]}) does not match U entries ({U_nodes.shape[0]}) in {ffile}")

    return {"nodes": nodes, "U_nodes": U_nodes}

In [5]:
def sample_velocity_at_particles(nodes, U_nodes, particle_pos):
    tree = cKDTree(nodes)
    _, idx = tree.query(particle_pos, k=1)
    return U_nodes[idx]

# ---------------- processing loop ----------------
frames = list(range(FRAME_START, FRAME_STOP + 1, FRAME_STEP))
metadata = {"frames": []}

for frame in frames:
    ppath = os.path.join(PFIELD_FOLDER, f"{PFIELD_BASENAME}.{frame}.h5")
    fpath = os.path.join(FDOM_FOLDER, f"{FDOM_BASENAME}.{frame}.h5")
    if not os.path.isfile(ppath):
        print(f"[skip] missing pfield file {ppath}")
        continue
    if not os.path.isfile(fpath):
        print(f"[skip] missing fdom file {fpath}")
        continue

    try:
        pdat = read_required_pfield(ppath)
        fdat = read_required_fdom(fpath)

        pos = pdat["pos"]
        Gamma_vec = pdat["Gamma_vec"]
        gamma_scalar = pdat["gamma_scalar"]
        sigma = pdat["sigma"]
        circulation = pdat["circulation"]
        vol = pdat["vol"]
        static = pdat["static"]
        idx_i = pdat["i"]
        C = pdat["C"]

        nodes = fdat["nodes"]
        U_nodes = fdat["U_nodes"]

        U_at_particles = sample_velocity_at_particles(nodes, U_nodes, pos)

        out = {
            "pos": pos,
            "Gamma_vec": Gamma_vec,
            "gamma_scalar": gamma_scalar,
            "sigma": sigma,
            "circulation": circulation,
            "vol": vol,
            "static": static,
            "i": idx_i,
            "C": C,
            "nodes": nodes,
            "U_nodes": U_nodes,
            "U_at_particles": U_at_particles,
            "frame": np.int32(frame),
            "source_files": np.array([ppath, fpath], dtype=object)
        }

        outname = os.path.join(OUT_FOLDER, f"frame_{frame}.npz")
        np.savez_compressed(outname, **out)
        print(f"[saved] {outname}  particles:{pos.shape} nodes:{nodes.shape} U:{U_nodes.shape}")

        metadata["frames"].append({
            "frame": int(frame),
            "n_particles": int(pos.shape[0]),
            "n_nodes": int(nodes.shape[0]),
            "out": outname
        })
    except Exception as e:
        print(f"[error frame {frame}] {e}")

# write metadata
meta_path = os.path.join(OUT_FOLDER, "metadata_frames.json")
with open(meta_path, "w") as fj:
    json.dump(metadata, fj, indent=2)
print(f"Done. Metadata -> {meta_path}")

[saved] training_pairs_clean/frame_100.npz  particles:(10250, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_110.npz  particles:(11270, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_120.npz  particles:(12290, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_130.npz  particles:(13310, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_140.npz  particles:(14330, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_150.npz  particles:(15350, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_160.npz  particles:(16370, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_170.npz  particles:(17390, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_180.npz  particles:(18410, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] training_pairs_clean/frame_190.npz  particles:(19430, 3) nodes:(2146689, 3) U:(2146689, 3)
[saved] tr

In [6]:
test_frame = 100   # change this as needed
test_file = f"{OUT_FOLDER}/frame_{test_frame}.npz"

print("\n=== Testing output file:", test_file, "===\n")

data = np.load(test_file, allow_pickle=True)

for key in data.files:
    arr = data[key]
    a = np.array(arr)

    print(f"\n--- {key} ---")
    print("shape:", a.shape, "dtype:", a.dtype)

    # Print 5 sample values (flattened or first rows)
    if a.ndim == 1:
        print("sample:", a[:5])
    elif a.ndim == 2:
        print("sample:\n", a[:5])
    elif a.ndim == 3:
        print("sample slice:\n", a[0, :5])
    else:
        print("sample (flattened):", a.flatten()[:5])

print("\nDone inspecting frame", test_frame)


=== Testing output file: training_pairs_clean/frame_100.npz ===


--- pos ---
shape: (10250, 3) dtype: float64
sample:
 [[ 1.0696796  -1.98331692 -0.47412186]
 [ 1.07061404 -1.98317728 -0.47394389]
 [ 1.0642922  -1.94370899 -0.4831904 ]
 [ 1.06522194 -1.94352151 -0.48291805]
 [ 1.05902163 -1.90599323 -0.49477918]]

--- Gamma_vec ---
shape: (10250, 3) dtype: float64
sample:
 [[-1.06241348e-04 -1.34600174e-04  3.89887759e-05]
 [-1.05909853e-04 -1.34706474e-04  3.89403879e-05]
 [-4.82787306e-05 -6.35089706e-05  1.68061283e-05]
 [-4.81272909e-05 -6.35358275e-05  1.67657561e-05]
 [-2.85809139e-05 -4.74402364e-05  1.12886102e-05]]

--- gamma_scalar ---
shape: (10250,) dtype: float64
sample: [1.75853790e-04 1.75724458e-04 8.15271189e-05 8.14501570e-05
 5.65232464e-05]

--- sigma ---
shape: (10250,) dtype: float64
sample: [0.10081795 0.10082182 0.10109631 0.10110108 0.10109972]

--- circulation ---
shape: (10250,) dtype: float64
sample: [0.20175232 0.20175232 0.09698475 0.09698475 0.06982339]